## Metrics Explanation  

This section provides a detailed explanation of the metrics calculated in this notebook for analyzing movement patterns based on tracking data grouped by **patients (`ID`)** and sperm cells (`tracker_id`).  

### 1. **VCL (Curvilinear Velocity)**  
- **Definition**: VCL represents the average velocity of a tracker along its actual curvilinear path. It measures the distance traveled over time, calculated for each sperm within a patient's dataset.  
- **Mathematical Formula**:  

$$v_{ci} = \frac{\|p_{i+1} - p_i\| + \|p_i - p_{i-1}\|}{2\Delta t}$$
$$\text{VCL} = \frac{1}{N-2} \sum_{i=1}^{N-1} v_{ci}$$

  Where:  
  - $p_i$: Position at frame $i$ (e.g., $(x_{center}, y_{center})$).  
  - $\|p_{i+1} - p_i\|$: Euclidean distance between positions at frames \(i+1\) and \(i\).  
  - $\|p_i - p_{i-1}\|$: Euclidean distance between positions at frames \(i\) and \(i-1\).  
  - $\Delta t$: Time interval between frames.  
  - $N$: Total number of frames for the tracker.

### 2. **VSL (Straight-Line Velocity)**  
- **Definition**: VSL measures the average velocity along a straight line connecting the initial and final positions of a sperm cell. It quantifies the efficiency of movement in terms of directness.  
- **Mathematical Formula**:  
  $$\text{VSL} = \frac{\|p_{\text{end}} - p_{\text{start}}\|}{T}$$  
  Where:  
  - $p_{\text{start}}$: Starting position.  
  - $p_{\text{end}}$: Ending position.  
  - $T$: Total time ($N \cdot \Delta t$).  

### 3. **VAP (Average Path Velocity)**  
- **Definition**: VAP represents the average velocity along the path taken by the sperm cell, providing a measure of its average speed over time.  
- **Mathematical Formula**:  
  $$\text{total\_distance} = \sum_{i=1}^{n-1} \sqrt{(x_i - x_{i-1})^2 + (y_i - y_{i-1})^2}$$  
  $$\text{total\_time} = (n - 1) \times \Delta t$$  
  $$\text{VAP} = \frac{\text{total\_distance}}{\text{total\_time}}$$  
  Where:  
  - $x_i$, $y_i$: Coordinates of the position at frame $i$.  
  - $n$: Total number of frames for the tracker.  
  - $\Delta t$: Time interval between frames.  

### 4. **ALH (Amplitude of Lateral Head Displacement)**  
- **Definition**: ALH quantifies the average deviation of the tracker’s position from its average path. It reflects the lateral movement relative to the trajectory.  
- **Mathematical Formula**:  
  $$\text{ALH} = \frac{1}{N} \sum_{i=1}^{N} \|\bar{p} - p_i\|$$  
  Where:  
  - $\bar{p}$: Mean position of the tracker over all frames (average path center).  
  - $p_i$: Position at frame $i$.  

### 5. **MAD (Mean Angular Displacement)**  
- **Definition**: MAD measures the average angular displacement between three consecutive positions. It is useful for analyzing the directional changes in movement.  
- **Mathematical Formula**:  
  $$\theta_i = \cos^{-1}\left(\frac{(p_i - p_{i-1}) \cdot (p_{i+1} - p_i)}{\|p_i - p_{i-1}\| \cdot \|p_{i+1} - p_i\|}\right)$$  
  $$\text{MAD} = \frac{1}{N-2} \sum_{i=2}^{N-1} |\theta_i|$$  
  Where:  
  - $\theta_i$: Angle between vectors formed by three consecutive points.  
  - $\cdot$: Dot product of two vectors.  
  - $\| \cdot \|$: Magnitude of a vector.  

---

### Notes  
- Metrics are computed for each tracker ID grouped by patient ID.  
- $N$: Total number of frames per sperm tracker.  
- All calculations assume the data is ordered temporally (e.g., by `ID`).  
- Metrics such as VCL, VSL, and VAP are expressed in units of distance per time (e.g., $\mu m/s$), while ALH is expressed in units of distance (e.g., $\mu m$), and MAD is in degrees.

# Imports

In [292]:
import pandas as pd
from nb_utils import set_root
import numpy as np
import json
import os

PROJECT_DIR = set_root(2)

# Parameters

In [293]:
path_data = PROJECT_DIR / "data"
path_intermediate = path_data / "02_intermediate"
path_primary = path_data / "03_primary"

file_path_data = path_primary / "tracker_cut.parquet"
file_path_horm = path_intermediate / "data_horm_concat.parquet"
tracker_columns = ["tracker_id",	"class_id",	"x_min",	"y_min",	"x_max",	"y_max",	"x_center",	"y_center"]

# Data

In [294]:
data = pd.read_parquet(file_path_data)
#data_horm = pd.read_parquet(file_path_horm)
data

,tracker_id,class_id,x_min,y_min,x_max,y_max,ID,x_center,y_center
0,0,0,81.517593,341.390228,98.150589,358.730377,6,89.834091,350.060303
1,1,0,220.177231,32.698105,235.510880,48.191841,6,227.844055,40.444973
2,2,0,501.025208,244.382629,518.151794,260.651978,6,509.588501,252.517303
3,3,0,168.232040,412.327515,191.797409,435.265625,6,180.014725,423.796570
4,4,0,445.648376,83.616653,466.867371,104.786850,6,456.257874,94.201752
...,...,...,...,...,...,...,...,...,...
2196284,2091,0,523.772156,161.258850,539.751770,181.233521,69,531.761963,171.246185
2196285,1829,0,461.298218,186.087952,476.646240,202.813721,69,468.972229,194.450836
2196286,3239,0,94.208496,424.543518,107.303009,438.854553,69,100.755753,431.699036
2196287,3254,0,435.511108,461.199219,446.554077,477.938171,69,441.032593,469.568695


# Functions

In [295]:
def calculate_metrics(df):
    """
    Calculates the VCL, VSL, VAP, ALH, and MAD metrics for each tracker_id grouped by patient ID.

    Args:
        df (pd.DataFrame): DataFrame with the columns ['ID', 'tracker_id', 'x_center', 'y_center'].

    Returns:
        pd.DataFrame: DataFrame containing the metrics per tracker_id grouped by patient ID.
    """
    results = []
    
    for patient_id, patient_group in df.groupby('ID'):
        
        for tracker_id, group in patient_group.groupby('tracker_id'):
            group = group.sort_values('ID')
            positions = group[['x_center', 'y_center']].values
            delta_t = 0.02  # time interval between frames

            # VCL
            vcl = np.mean([
                (np.linalg.norm(positions[i] - positions[i - 1]) + np.linalg.norm(positions[i + 1] - positions[i])) / (2 * delta_t)
                for i in range(1, len(positions) - 1)
            ]) if len(positions) > 2 else 0

            # VSL
            vsl = (
                np.linalg.norm(positions[-1] - positions[0]) / (len(positions) * delta_t)
                if len(positions) > 1 else 0
            )

            # VAP
            total_distance = sum(np.linalg.norm(positions[i] - positions[i - 1]) for i in range(1, len(positions)))
            total_time = (len(positions) - 1) * delta_t
            vap = total_distance / total_time if total_time > 0 else 0

            # ALH
            avg_path = np.mean(positions, axis=0)
            alh = np.mean([
                np.linalg.norm(positions[i] - avg_path)
                for i in range(len(positions))
            ]) if len(positions) > 1 else 0

            # MAD
            mad = np.mean([
                np.abs(np.degrees(np.arccos(
                    np.clip(
                        np.dot(positions[i] - positions[i - 1], positions[i + 1] - positions[i]) /
                        (np.linalg.norm(positions[i] - positions[i - 1]) *
                         np.linalg.norm(positions[i + 1] - positions[i])),
                        -1.0, 1.0
                    )
                )))
                for i in range(1, len(positions) - 1)
            ]) if len(positions) > 2 else 0

            results.append({
                'ID': patient_id,
                'tracker_id': tracker_id,
                'VCL': vcl,
                'VSL': vsl,
                'VAP': vap,
                'ALH': alh,
                'MAD': mad
            })

    metrics_df = pd.DataFrame(results)
    return metrics_df


def calculate_metrics_per_second(df, window_seconds=1):
    """
    Calculates the average metrics (VCL, VSL, VAP, ALH, MAD) per window_seconds and the average position (x, y)
    for each tracker_id grouped by patient ID.

    Args:
        df (pd.DataFrame): DataFrame with the columns ['ID', 'tracker_id', 'x_center', 'y_center'].
        window_seconds (int): The length of the time window in seconds.

    Returns:
        pd.DataFrame: DataFrame containing the average metrics per window_seconds with the average position.
    """
    results = []
    frames_per_second = 50 * window_seconds
    delta_t = 20e-3 * window_seconds  # Adjust based on frame rate and window size

    for patient_id, patient_group in df.groupby('ID'):
        for tracker_id, group in patient_group.groupby('tracker_id'):
            group = group.sort_values('ID').reset_index(drop=True)
            num_windows = len(group) // frames_per_second

            for window in range(num_windows):
                start_frame = window * frames_per_second
                end_frame = start_frame + frames_per_second
                subset = group.iloc[start_frame:end_frame]

                if len(subset) > 0:
                    positions = subset[['x_center', 'y_center']].values

                    # Average positions
                    avg_x = np.mean(subset['x_center'])
                    avg_y = np.mean(subset['y_center'])

                    # Scale the average positions
                    scaled_x = 100 + ((avg_x - np.min(subset['x_center'])) * (1200 - 100)) / (np.max(subset['x_center']) - np.min(subset['x_center']))
                    scaled_y = 0 + ((avg_y - np.min(subset['y_center'])) * (500 - 0)) / (np.max(subset['y_center']) - np.min(subset['y_center']))

                    # VCL
                    vcl = np.mean([
                        (np.linalg.norm(positions[i] - positions[i - 1]) + np.linalg.norm(positions[i + 1] - positions[i])) / (2 * delta_t)
                        for i in range(1, len(positions) - 1)
                    ]) if len(positions) > 2 else 0

                    # VSL
                    vsl = (
                        np.linalg.norm(positions[-1] - positions[0]) / (len(positions) * delta_t)
                        if len(positions) > 1 else 0
                    )

                    # VAP
                    total_distance = sum(np.linalg.norm(positions[i] - positions[i - 1]) for i in range(1, len(positions)))
                    total_time = (len(positions) - 1) * delta_t
                    vap = total_distance / total_time if total_time > 0 else 0

                    # ALH
                    avg_path = np.mean(positions, axis=0)
                    alh = np.mean([
                        np.linalg.norm(positions[i] - avg_path)
                        for i in range(len(positions))
                    ]) if len(positions) > 1 else 0

                    # MAD
                    mad = np.mean([
                        np.abs(np.degrees(np.arccos(
                            np.clip(
                                np.dot(positions[i] - positions[i - 1], positions[i + 1] - positions[i]) /
                                (np.linalg.norm(positions[i] - positions[i - 1]) *
                                np.linalg.norm(positions[i + 1] - positions[i])),
                                -1.0, 1.0
                            )
                        )))
                        for i in range(1, len(positions) - 1)
                    ]) if len(positions) > 2 else 0

                    results.append({
                        'ID': patient_id,
                        'tracker_id': tracker_id,
                        'window': window,
                        'x': scaled_x,
                        'y': scaled_y,
                        'VCL': vcl,
                        'VSL': vsl,
                        'VAP': vap,
                        'ALH': alh,
                        'MAD': mad
                    })

    metrics_per_window_df = pd.DataFrame(results)
    return metrics_per_window_df



def metrics_dataframe_to_json(metrics_df):
    """
    Converts the metrics dataframe to a specified JSON format.

    Args:
        metrics_df (pd.DataFrame): DataFrame containing metrics grouped by patient ID and tracker_id.

    Returns:
        str: JSON formatted string.
    """
    data = {'metrics': []}
    for patient_id, patient_group in metrics_df.groupby('ID'):
        patient_metrics = {'id': int(patient_id), 'trackers': []}
        for _, row in patient_group.iterrows():
            tracker_metrics = {
                'tracker_id': int(row['tracker_id']),
                'VCL': int(row['VCL']),
                'VSL': int(row['VSL']),
                'VAP': int(row['VAP']),
                'ALH': int(row['ALH']),
                'MAD': int(row['MAD'])
            }
            patient_metrics['trackers'].append(tracker_metrics)
        data['metrics'].append(patient_metrics)
    return json.dumps(data, indent=4)


def dataframe_to_json(df):
    """
    Converts the dataframe to the specified JSON format.

    Args:
        df (pd.DataFrame): DataFrame with metrics and positions.

    Returns:
        str: JSON formatted string.
    """
    data = {'individuos': []}
    for patient_id, patient_group in df.groupby('ID'):
        individual = {'id': int(patient_id), 'espermatozoides': []}
        for tracker_id, group in patient_group.groupby('tracker_id'):
            espermatozoide = {'id': int(tracker_id), 'route': [], 'frames': []}
            for _, row in group.iterrows():
                espermatozoide['route'].append({'x': int(row['x']), 'y': int(row['y'])})
                espermatozoide['frames'].append({
                    'x': int(row['x']),
                    'y': int(row['y']),
                    'VCL': int(row['VCL']),
                    'VSL': int(row['VSL']),
                    'VAP': int(row['VAP']),
                    'ALH': int(row['ALH']),
                    'MAD': int(row['MAD'])
                })
            individual['espermatozoides'].append(espermatozoide)
        data['individuos'].append(individual)
    return json.dumps(data, indent=4)

def save_json_to_directory(json_data, filename):
    """
    Saves the JSON data to a specific directory 'visualization/outputs' located 
    one level up from the current working directory.

    Args:
        json_data (str): JSON formatted string.
        filename (str): Name of the JSON file.
    """
    parent_directory = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
    target_directory = os.path.join(parent_directory, 'visualization', 'outputs')
    
    if not os.path.exists(target_directory):
        os.makedirs(target_directory)
    
    file_path = os.path.join(target_directory, filename)
    

    with open(file_path, 'w') as f:
        f.write(json_data)
    
    print(f"JSON saved to {file_path}")


# Calculate metrics

In [301]:
metrics_df = calculate_metrics(data)
metrics_df

,ID,tracker_id,VCL,VSL,VAP,ALH,MAD
0,11,0,24.236023,1.295533,25.058355,2.319782,110.747330
1,11,1,13.467506,0.091070,13.486833,0.256387,118.706230
2,11,2,31.063055,0.638578,31.015694,1.302806,112.631401
3,11,3,34.240788,1.952736,35.497402,2.455897,114.107567
4,11,4,41.229862,3.950342,43.357918,1.757048,103.840889
...,...,...,...,...,...,...,...
1072,48,523,0.000000,0.000000,0.000000,0.000000,0.000000
1073,48,524,40.036640,5.577832,40.036640,0.356231,177.885483
1074,48,525,61.260101,36.900230,62.111813,1.156447,91.426445
1075,48,526,0.000000,0.000000,0.000000,0.000000,0.000000


In [303]:
metrics_per_second_df = calculate_metrics_per_second(data, window_seconds=2) #coloquei uma janela de 2 segundo pq achei que 1 segundo iria ficar com muita coisa
metrics_per_second_df

,ID,tracker_id,window,x,y,VCL,VSL,VAP,ALH,MAD
0,11,0,0,743.415527,266.213776,16.424744,1.816250,16.829716,2.181424,104.386757
1,11,0,1,682.373901,173.436874,10.618831,1.951531,10.620710,2.852505,109.556122
2,11,1,0,614.326233,257.536316,5.917129,0.132890,5.966982,0.254748,115.922165
3,11,1,1,582.319763,168.025818,7.682483,0.068652,7.681798,0.263850,121.056046
4,11,2,0,836.807251,304.262756,22.232698,1.320314,22.138752,2.089112,109.958374
...,...,...,...,...,...,...,...,...,...,...
220,48,276,0,576.489746,219.550049,176.818954,15.605796,189.643921,37.167763,55.633648
221,48,295,0,509.173523,272.480255,57.066486,0.502919,56.971817,2.376691,112.721840
222,48,302,0,886.802856,120.628143,16.831068,2.077834,17.651899,1.822940,103.681587
223,48,310,0,686.009338,231.182373,214.427795,13.204291,229.723679,46.516148,75.254074


In [ ]:
metrics_per_second_df = metrics_per_second_df.groupby(["ID", "tracker_id"]).filter(lambda x: len(x) > 1)

# Data to JSON

In [305]:
#json com valores das métricas (no caso, valores gerais)
json_dt = metrics_dataframe_to_json(metrics_df)
save_json_to_directory(json_dt, 'metrics_general.json')

#json de acordo com a janela de tempo que foi definida no início
json_data = dataframe_to_json(metrics_per_second_df)
save_json_to_directory(json_data, 'data_window.json')


JSON saved to /home/manuel/projects/repos/cin-dataviz/visualization/outputs/metrics_general.json
JSON saved to /home/manuel/projects/repos/cin-dataviz/visualization/outputs/data_window.json
